In [0]:
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True, inferSchema=True).toPandas()
df_races   = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv",   header=True, inferSchema=True).toPandas()
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True, inferSchema=True).toPandas()

display(df_results.head())

resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1
2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1
3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1
4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1
5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1


In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv",
                    header=True, inferSchema=True).toPandas()

df = df[['grid', 'laps', 'statusId', 'points', 'positionOrder']].dropna()
df = df[df['positionOrder'] > 0]  

X = df[['grid', 'laps', 'statusId', 'points']]
y = df['positionOrder']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"testsize: {X_train.shape}, testsize: {X_test.shape}")
display(df.head())

testsize: (21407, 4), testsize: (5352, 4)


grid,laps,statusId,points,positionOrder
1,58,1,10.0,1
5,58,1,8.0,2
7,58,1,6.0,3
11,58,1,5.0,4
3,58,1,4.0,5


In [0]:
import re, subprocess

username = spark.sql("SELECT current_user()").collect()[0][0]
experiment_path = f"/Users/{username}/F1_Position_Prediction"
mlflow.set_experiment(experiment_path)
print(f"Experimentpath: {experiment_path}")

Experimentpath: /Users/ww2774@columbia.edu/F1_Position_Prediction


In [0]:
def run_experiment(params, run_name):
    with mlflow.start_run(run_name=run_name):

        # 1. 
        model = RandomForestRegressor(
            n_estimators   = params['n_estimators'],
            max_depth      = params['max_depth'],
            min_samples_split = params.get('min_samples_split', 2),
            min_samples_leaf  = params.get('min_samples_leaf', 1),
            random_state=42
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # 2. Log 
        mlflow.log_params(params)

        # 3. Log metrics
        mae  = mean_absolute_error(y_test, y_pred)
        mse  = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_test, y_pred)

        mlflow.log_metric("mae",  mae)
        mlflow.log_metric("mse",  mse)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2",   r2)

        # 4. Log
        mlflow.sklearn.log_model(model, "random-forest-model")

        # 5a. Artifact: Residuals
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.scatter(y_pred, y_test - y_pred, alpha=0.4, color='steelblue')
        ax.axhline(0, color='red', linestyle='--')
        ax.set_xlabel("Predicted Position")
        ax.set_ylabel("Residuals")
        ax.set_title(f"Residuals - {run_name}")
        fig.savefig("/tmp/residuals.png", bbox_inches='tight')
        mlflow.log_artifact("/tmp/residuals.png", "residuals_png")
        plt.close()

        # 5b. Artifact: Feature Importance CSV
        feat_imp = pd.DataFrame({
            'feature':    X_train.columns,
            'importance': model.feature_importances_
        }).sort_values('importance', ascending=False)

        feat_imp.to_csv("/tmp/feature_importance.csv", index=False)
        mlflow.log_artifact("/tmp/feature_importance.csv", "feature-importance")

        fig2, ax2 = plt.subplots(figsize=(6, 4))
        sns.barplot(data=feat_imp, x='importance', y='feature', ax=ax2)
        ax2.set_title("Feature Importances")
        fig2.savefig("/tmp/feature_importance_plot.png", bbox_inches='tight')
        mlflow.log_artifact("/tmp/feature_importance_plot.png", "feature-importance")
        plt.close()

        print(f"{run_name:12s} | n_est={params['n_estimators']:4d} "
              f"| depth={str(params['max_depth']):4s} "
              f"| MAE={mae:.2f} | RMSE={rmse:.2f} | R²={r2:.3f}")

In [0]:
experiments = [
    {"n_estimators": 50,   "max_depth": 3,    "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 100,  "max_depth": 5,    "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 200,  "max_depth": 8,    "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 300,  "max_depth": 10,   "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 500,  "max_depth": 10,   "min_samples_split": 3, "min_samples_leaf": 1},
    {"n_estimators": 1000, "max_depth": 10,   "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 200,  "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 400,  "max_depth": 15,   "min_samples_split": 4, "min_samples_leaf": 2},
    {"n_estimators": 600,  "max_depth": 12,   "min_samples_split": 3, "min_samples_leaf": 2},
    {"n_estimators": 750,  "max_depth": 20,   "min_samples_split": 2, "min_samples_leaf": 1},
]

for i, params in enumerate(experiments):
    run_experiment(params, run_name=f"Run_{i+1}")

print("\ndone")

2026/04/13 18:01:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-f11e6b56-7522.cloud.databricks.com/ml/experiments/1315616465199883/models/m-2b449570010543738910cf0d49fa8ecf?o=7474645003041195
2026/04/13 18:01:11 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.


Run_1        | n_est=  50 | depth=3    | MAE=2.54 | RMSE=3.33 | R²=0.813


2026/04/13 18:01:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-f11e6b56-7522.cloud.databricks.com/ml/experiments/1315616465199883/models/m-a309d803718f4695aeed09dd0f2f9a7a?o=7474645003041195
2026/04/13 18:01:18 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.


Run_2        | n_est= 100 | depth=5    | MAE=2.32 | RMSE=3.08 | R²=0.840


2026/04/13 18:01:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-f11e6b56-7522.cloud.databricks.com/ml/experiments/1315616465199883/models/m-f641ab418cc94005b433ad43fa907b28?o=7474645003041195
2026/04/13 18:01:27 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.


...

Run_8        | n_est= 400 | depth=15   | MAE=2.05 | RMSE=2.84 | R²=0.864


2026/04/13 18:03:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-f11e6b56-7522.cloud.databricks.com/ml/experiments/1315616465199883/models/m-2fa032d0878e416795f103470f383e8c?o=7474645003041195
2026/04/13 18:03:24 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.


Run_9        | n_est= 600 | depth=12   | MAE=2.04 | RMSE=2.81 | R²=0.867


2026/04/13 18:03:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-f11e6b56-7522.cloud.databricks.com/ml/experiments/1315616465199883/models/m-50cc69680b33474396a991149e40a2dc?o=7474645003041195
2026/04/13 18:04:00 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.


Run_10       | n_est= 750 | depth=20   | MAE=2.11 | RMSE=2.97 | R²=0.851

done


In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
exp    = client.get_experiment_by_name(experiment_path)

best_runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["metrics.r2 DESC"],
    max_results=3
)

print("Top 3:\n")
for i, run in enumerate(best_runs):
    p = run.data.params
    m = run.data.metrics
    print(f"#{i+1} Run: {run.info.run_name}")
    print(f"   n_estimators={p['n_estimators']}, max_depth={p['max_depth']}")
    print(f"   R²={m['r2']:.3f}, MAE={m['mae']:.2f}, RMSE={m['rmse']:.2f}\n")

Top 3:

#1 Run: Run_9
   n_estimators=600, max_depth=12
   R²=0.867, MAE=2.04, RMSE=2.81

#2 Run: Run_6
   n_estimators=1000, max_depth=10
   R²=0.866, MAE=2.06, RMSE=2.82

#3 Run: Run_4
   n_estimators=300, max_depth=10
   R²=0.866, MAE=2.06, RMSE=2.82



## Best Model: Run_9

Parameters: n_estimators=600, max_depth=12, min_samples_split=3, min_samples_leaf=2

Run_9 achieved the best performance with R²=0.867, MAE=2.04, RMSE=2.81.

### Why this is the best:

R²=0.867 means the model explains 86.7% of variance in finishing 
position — the highest across all runs.
MAE=2.04means predictions are off by only ~2 positions on average, 
better than all other runs.
max_depth=12 strikes the right balance: deep enough to capture 
complex patterns (unlike Run_1's depth=3 which underfits, R²=0.813), 
but constrained enough to avoid overfitting (unlike Run_7's unlimited 
depth which actually performed worse at R²=0.848).
n_estimators=600provides enough trees for stable predictions. 
Notably, increasing to 1000 trees (Run_6) gave no improvement, 
confirming 600 is the sweet spot — adding more trees beyond this point 
yields diminishing returns.
Compared to Run_10 (depth=20, R²=0.851), the shallower depth=12 of 
Run_9 generalizes better on the test set, indicating Run_10 starts to 
overfit.